In [8]:
import numpy as np
import torch
import os
import sys
import json
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import pickle

sys.path.append(os.path.abspath('../utils'))
from math_utils import *

In [3]:
def load_num(num, mirror=False):
    path = f"/home/amir.mann/MDM/dataset/HumanML3D/new_joints/{num:06}.npy"
    with open(path, "rb") as f:
        data = np.load(f)
    return torch.from_numpy(data)

load_num(0).shape

torch.Size([116, 22, 3])

In [4]:
data_2d = perspective_projection(load_num(0), [0], [0], 1)
first_frame = data_2d[0, 0, ...]
first_frame.shape

torch.Size([22, 2])

In [2]:
connections = [
    [21, 19, 17, 14],
    [20, 18, 16, 13],
    [11, 8, 5, 2, 0],
    [10, 7, 4, 1, 0],
    [0, 3, 6, 9, 12, 15],
]

In [9]:
with open("/home/amir.mann/temp/a.pkl", "rb") as f:
    a = pickle.load(f)
for k in a.keys():
    exec(f"{k} = a['{k}']")

In [81]:
def plot_frame(frame):
    points = frame.numpy()
    # Plot the points
    plt.scatter(points[:, 0], points[:, 1], c='blue', marker='o', label='Points')

    # Connect the points based on the connections
    for path in connections:
        for i in range(len(path) - 1):
            point1 = points[path[i]]
            point2 = points[path[i + 1]]
            plt.plot([point1[0], point2[0]], [point1[1], point2[1]], c='black', lw=1)

    # Add labels and title
    plt.xlabel("X-axis")
    plt.ylabel("Y-axis")
    plt.title("2D Points Visualization with Connections")
    plt.legend()

    # Number each point in the plot
    for i, (x, y) in enumerate(points):
        plt.text(x, y, str(i), fontsize=9, ha='right')
    
    # Set fixed axis limits
    plt.xlim(-3, 3)
    plt.ylim(-1, 1)
    
    # Show the plot
    plt.grid(True)
    plt.show()

angles = 3
print(load_num(0).shape)
print(target_xyz.shape)
target_xyz_28 = target_xyz[28, :, :, :].unsqueeze(0)
cam_hor_angles_28 = cam_hor_angles[28].unsqueeze(0)
s = compute_into_camera_shift_from_angle(target_xyz_28, cam_hor_angles_28)
sin_hor = - torch.sin(cam_hor_angles_28)
cos_hor = - torch.cos(cam_hor_angles_28)

print(s.shape)
print(cam_hor_angles_28)
target_xyz_28_after_shift = target_xyz_28 - s.unsqueeze(1).unsqueeze(3)
x = target_xyz_28_after_shift[:, :, 0, :]  # Shape: [batch_size, njoints, nframes]
y = target_xyz_28_after_shift[:, :, 1, :]  # Shape: [batch_size, njoints, nframes]
z = target_xyz_28_after_shift[:, :, 2, :]  # Shape: [batch_size, njoints, nframes]
transformed_values = x * sin_hor[:, None, None] + z * cos_hor[:, None, None]  # Shape: [batch_size, njoints, nframes]
print("transformed_values", torch.max(transformed_values))
we, wa = perspective_projection_batch(target_xyz_28_after_shift, cam_hor_angles_28, cam_ver_angles[28].unsqueeze(0), cam_distance[28].unsqueeze(0))

we[0, 9, 0, 171]
cam_ver_angles[28]
print(we[0, :, :, 171].shape)

print("wa min", torch.min(wa))
print(first_frame.shape)
print(target_xyz_28_after_shift[0, :, :2, 171].shape)
print(sin_hor)
print(cos_hor)
print(target_xyz_28_after_shift[0, :, :, 171])
cam_ver_angles_28 = cam_ver_angles[28].unsqueeze(0)
print(cam_hor_angles_28, cam_ver_angles_28)
rotation_mat = batch_rotation_matrix(cam_hor_angles_28, cam_ver_angles_28)
print(rotation_mat)
#plot_frame(target_xyz_28_after_shift[0, :, [2, 1], 171])
#plot_frame(target_xyz_28_after_shift[0, :, :2, 171])


torch.Size([116, 22, 3])
torch.Size([64, 22, 3, 196])
torch.Size([1, 3])
tensor([1.7616])
transformed_values tensor(0.)
torch.Size([22, 2])
wa min tensor(-0.7157)
torch.Size([22, 2])
torch.Size([22, 2])
tensor([-0.9819])
tensor([0.1896])
tensor([[ 2.2708,  0.0348, -1.5809],
        [ 2.2690, -0.0501, -1.6392],
        [ 2.2677, -0.0490, -1.5100],
        [ 2.2401,  0.1626, -1.5881],
        [ 2.4476, -0.3894, -1.7283],
        [ 2.2805, -0.4385, -1.4903],
        [ 2.2583,  0.3045, -1.5913],
        [ 2.3725, -0.8149, -1.7127],
        [ 2.1748, -0.8458, -1.5546],
        [ 2.2867,  0.3542, -1.5956],
        [ 2.4767, -0.8702, -1.7942],
        [ 2.3010, -0.8942, -1.4909],
        [ 2.2644,  0.5718, -1.6123],
        [ 2.2555,  0.4627, -1.6740],
        [ 2.2736,  0.4808, -1.5294],
        [ 2.3398,  0.6396, -1.6304],
        [ 2.2098,  0.4729, -1.7970],
        [ 2.2516,  0.5010, -1.4101],
        [ 2.0432,  0.3468, -1.9464],
        [ 2.1367,  0.3758, -1.2093],
        [ 2.1329,  0.1

In [85]:
for axis in "xyz":
    for angle in torch.tensor([0, 1, 2, 3, 4, 5, 6]):
        # Single matrix
        single_matrix = axis_rotation_matrix(angle, axis)

        # Batch matrix (batch size = 1)
        batch_matrix = batch_axis_rotation_matrices(angle.unsqueeze(0), axis)[0]

        # Verify
        assert torch.allclose(single_matrix, batch_matrix), "The matrices do not match!"
print("The matrices match!")



The matrices match!
